In [1]:
# Cell 1 — Imports (headless-safe)
import matplotlib

matplotlib.use("Agg")  # must come before pyplot import for headless CI

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Imports OK — output dir:", OUTPUT_DIR.resolve())

Matplotlib is building the font cache; this may take a moment.


Imports OK — output dir: C:\Users\Admin\expense-project\expense-ai\notebooks\output


In [2]:
# Cell 2 — Load model and encode corpus
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
model = SentenceTransformer(MODEL_NAME)

corpus = [
    "Coffee shop purchase at downtown café",
    "Monthly SaaS subscription renewal for project management tool",
    "Round-trip airline ticket to New York for client meeting",
    "Office supplies order: pens, paper, and printer cartridges",
    "Team lunch expense at Italian restaurant",
    "Cloud hosting invoice from AWS for production servers",
    "Hotel accommodation for three-night business trip",
    "Ride-share fare from airport to headquarters",
    "Annual software licence renewal for design suite",
    "Grocery run for office kitchen snacks and beverages",
]

# Short labels for scatter plot annotations
labels = [
    "Coffee",
    "SaaS sub",
    "Flight",
    "Office supplies",
    "Team lunch",
    "Cloud hosting",
    "Hotel",
    "Ride-share",
    "SW licence",
    "Groceries",
]

embeddings = model.encode(corpus, show_progress_bar=True)
print(f"Encoded {len(corpus)} sentences → shape {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Admin\expense-project\expense-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoded 10 sentences → shape (10, 768)


In [3]:
# Cell 3 — Viz 1: 2-D PCA scatter
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings)  # shape (10, 2)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=120, zorder=3)

for i, label in enumerate(labels):
    ax.annotate(
        label,
        xy=(coords[i, 0], coords[i, 1]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=9,
    )

var = pca.explained_variance_ratio_
ax.set_xlabel(f"PC1 ({var[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({var[1]:.1%} variance)")
ax.set_title("PCA of Expense Transaction Embeddings (all-mpnet-base-v2)")
ax.grid(True, alpha=0.3)
fig.tight_layout()

out_path = OUTPUT_DIR / "pca_scatter.png"
fig.savefig(out_path, dpi=150)
print(f"Saved → {out_path}")
plt.close(fig)

Saved → output\pca_scatter.png


In [4]:
# Cell 4 — Viz 2: cosine-similarity heatmap (10×10)
sim_matrix = cosine_similarity(embeddings)  # shape (10, 10)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, vmin=0.0, vmax=1.0, cmap="viridis")
fig.colorbar(im, ax=ax, label="Cosine similarity")

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(labels, fontsize=8)

# Annotate each cell with its value
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(
            j,
            i,
            f"{sim_matrix[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=7,
            color="white" if sim_matrix[i, j] < 0.6 else "black",
        )

ax.set_title("Cosine Similarity Heatmap — Expense Corpus")
fig.tight_layout()

out_path = OUTPUT_DIR / "cosine_heatmap.png"
fig.savefig(out_path, dpi=150)
print(f"Saved → {out_path}")
plt.close(fig)

Saved → output\cosine_heatmap.png


In [5]:
# Cell 5 — Viz 3: top-3 retrieval bar chart for a sample query
QUERY = "monthly software subscription payment"
query_emb = model.encode([QUERY])  # shape (1, dim)

scores = cosine_similarity(query_emb, embeddings)[0]  # shape (10,)
top3_idx = np.argsort(scores)[::-1][:3]

top3_labels = [labels[i] for i in top3_idx]
top3_scores = [float(scores[i]) for i in top3_idx]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(top3_labels[::-1], top3_scores[::-1], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.bar_label(bars, fmt="{:.3f}", padding=4, fontsize=10)
ax.set_xlim(0, 1.0)
ax.set_xlabel("Cosine Similarity")
ax.set_title(f'Top-3 Retrieval for: "{QUERY}"')
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

out_path = OUTPUT_DIR / "top3_retrieval.png"
fig.savefig(out_path, dpi=150)
print(f"Saved → {out_path}")
print("\nTop-3 results:")
for rank, (lbl, sc) in enumerate(zip(top3_labels, top3_scores, strict=True), 1):
    print(f"  {rank}. {lbl:20s}  score={sc:.4f}")
plt.close(fig)

Saved → output\top3_retrieval.png

Top-3 results:
  1. SaaS sub              score=0.6666
  2. SW licence            score=0.5825
  3. Cloud hosting         score=0.3342
